### Ноутбук "EDA и подготовка данных"

#### Описание

Генерация и загрузка синтетических данных в PostgreSQL, первичная проверка целостности перед основным анализом.

##### Импорт модулей и библиотек

In [ ]:
import sys
sys.path.append("../src")

from generate_data import generate_users, generate_subscription

In [ ]:
import pandas as pd
import numpy as np
from sqlalchemy import text

from db_connection import get_engine
engine = get_engine()

##### Создание таблицы "users"

Генерация датафрейма "users_df" с 3000 пользователей, без столбца "id"

In [ ]:
users_df = generate_users(3000)
display(users_df.head())
display(users_df.info())

,acquisition_channel,plan,country,signup_date
0,organic,free,Russia,2025-04-15
1,organic,pro,Russia,2025-07-09
2,organic,free,Belarus,2025-09-04
3,paid_search,basic,Belarus,2025-04-07
4,social,free,Belarus,2025-04-25


<class 'pandas.DataFrame'>
RangeIndex: 3000 entries, 0 to 2999
Data columns (total 4 columns):
 #   Column               Non-Null Count  Dtype 
---  ------               --------------  ----- 
 0   acquisition_channel  3000 non-null   str   
 1   plan                 3000 non-null   str   
 2   country              3000 non-null   str   
 3   signup_date          3000 non-null   object
dtypes: object(1), str(3)
memory usage: 93.9+ KB


None

*Очистка таблицы "users"*

In [ ]:
with engine.begin() as conn:
    conn.execute(text("TRUNCATE TABLE users RESTART IDENTITY CASCADE;"))

Генерация датафрейма "users_db" со столбцом "id"

In [ ]:
users_df.to_sql("users", engine, if_exists="append", index=False)

users_db = pd.read_sql("SELECT * FROM users;", engine)
display(users_db.columns)
display(users_db.shape)

Index(['id', 'acquisition_channel', 'plan', 'country', 'signup_date'], dtype='str')

(3000, 5)

##### Создание таблицы "subscription"

Генерация датафрейма "subscriptions_df" без столбца "id"

In [ ]:
subscriptions_df = generate_subscription(users_db)
display(subscriptions_df.head())
display(subscriptions_df.info())

,user_id,plan,price,start_date,end_date,status
0,2,pro,250.0,2025-07-19,2025-09-20,canceled
1,4,basic,100.0,2025-04-10,NaT,active
2,7,basic,100.0,2025-04-21,NaT,active
3,10,basic,100.0,2026-07-28,2026-09-08,canceled
4,12,basic,100.0,2026-08-08,NaT,active


<class 'pandas.DataFrame'>
RangeIndex: 1404 entries, 0 to 1403
Data columns (total 6 columns):
 #   Column      Non-Null Count  Dtype         
---  ------      --------------  -----         
 0   user_id     1404 non-null   int64         
 1   plan        1404 non-null   str           
 2   price       1404 non-null   float64       
 3   start_date  1404 non-null   datetime64[us]
 4   end_date    417 non-null    datetime64[us]
 5   status      1404 non-null   str           
dtypes: datetime64[us](2), float64(1), int64(1), str(2)
memory usage: 65.9 KB


None

Генерация датафрейма "subscriptions_db" со столбцом "id"

In [ ]:
subscriptions_df.to_sql("subscriptions", engine, if_exists="append", index=False)

subscriptions_db = pd.read_sql("SELECT * FROM subscriptions;", engine)
display(subscriptions_db.columns)
display(subscriptions_db.shape)

Index(['id', 'user_id', 'plan', 'price', 'start_date', 'end_date', 'status'], dtype='str')

(1404, 7)